# Generate entity-parity Mid/Low rewrites

One call per event group. Each call receives the High, Mid and Low responses for a single event and
returns **both** a new Mid and a new Low as JSON, which is parsed and written into a `parity_answer`
column placed directly after `cleaned_answer`.

250 groups x 2 conditions = **500 calls**, not 1500 — the three levels of an event must be seen together
so the model can hold the register ordering while lifting the content.

**The roles are inverted on purpose.** High is the *content source*; the existing Mid and Low are
*register exemplars only*. Asking a model to "improve" an existing Mid makes it anchor on that text and
lightly edit, inheriting the very omissions being targeted. Sourcing all facts from High and using
Mid/Low only to show the target reading level is what produces the dissociation.

**What success looks like.** FK grade and formality still descend High > Mid > Low, while entity
coverage stays flat near 1.0. Every metric moving together in either direction is a failure: flat FK
means the register adaptation was deleted, falling coverage means nothing was fixed.

Prompt construction, JSON parsing and CSV writing live in `src/parity_rewrite.py`, so they are testable
without a GPU. This notebook is the driver.

In [11]:
from pathlib import Path
import os, sys, time, gc, json

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from parity_rewrite import (
    SYSTEM_PROMPT, build_user_prompt, parse_response,
    load_groups, existing_parity, write_parity_column,
    coverage, fk_grade, LEVELS,
    tone_ok, informality_score, retry_note, reset_column, TONE_MIN,
)

ANSWER_DIR = ROOT / "newly_generated_answers"

print("root  :", ROOT)
print("torch :", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("gpu   :", p.name, f"| {p.total_memory/1024**3:.1f} GiB")

root  : /home/anirban/Desktop/Research/qwen_before_after_audit_package_fixed_v2/qwen_before_after_audit
torch : 2.14.0+cu130 | cuda: True
gpu   : NVIDIA GeForce RTX 5060 Ti | 15.5 GiB


## 1. Configuration

`MAX_NEW_TOKENS = 3072` because the output holds *two* responses, each of which may run as long as the
High response it is built from (implicit High reaches ~1100 words at p95). Input prompts reach ~5.1k
tokens on the implicit side, so a batch of 2 is the safe default on 16 GB — the loop halves the batch
automatically on OOM.

Temperature is 0.3, not the 0.7 used for the audit generation. This is a constrained rewriting task, not
free generation; low temperature keeps entities verbatim and JSON well-formed.

**Model choice matters here.** These outputs become fine-tuning targets, so their quality caps what the
fine-tuned model can learn. The local 4B can do this — it is handed the content and only has to re-register
it — but a larger local model in 4-bit (`Qwen/Qwen3-14B-Instruct`, `LOAD_IN_4BIT = True`) will produce
better targets and still fit 16 GB. Self-distillation with the 4B is the cheapest path and a valid
baseline, just expect a lower ceiling.

In [12]:
BASE_MODEL   = "Qwen/Qwen3-4B-Instruct-2507"
LOAD_IN_4BIT = False

MAX_NEW_TOKENS  = 3072
GEN_TEMPERATURE = 0.3
GEN_TOP_P       = 0.9
GEN_TOP_K       = 20
SEED            = 42

BATCH_SIZE   = 2
MAX_PASSES   = 3            # retries now carry corrective feedback, so they can actually converge
COVERAGE_MIN = 0.85         # entity-proxy coverage a rewrite must reach

SOURCE_COLUMN = "cleaned_answer"
NEW_COLUMN    = "parity_answer"
BACKUP_COLUMN = "parity_answer_v1"   # the first batch is copied here before regeneration

CHECKPOINT_EVERY = 10
LOG_EVERY_ROW    = True
PREVIEW_CHARS    = 80

SMOKE_TEST = True
SMOKE_ROWS = 4

print(f"{BASE_MODEL} | max_new={MAX_NEW_TOKENS} | batch={BATCH_SIZE} | "
      f"coverage>={COVERAGE_MIN} | tone>={TONE_MIN} | passes<={MAX_PASSES} | smoke={SMOKE_TEST}")

Qwen/Qwen3-4B-Instruct-2507 | max_new=3072 | batch=2 | coverage>=0.85 | passes<=3 | smoke=False


## 2. Load the event groups

Each group is one event with all three level responses present.

In [13]:
FILES = {
    "explicit": ANSWER_DIR / "explicit_answers_local_base.csv",
    "implicit": ANSWER_DIR / "implicit_answers_local_base.csv",
}

groups = {}
for cond, path in FILES.items():
    g = load_groups(path, SOURCE_COLUMN)
    groups[cond] = g
    lens = sorted(len(build_user_prompt(x["Topic"], x["Source_Law"],
                                        x["High Literacy"], x["Mid Literacy"], x["Low Literacy"])) // 4
                  for x in g)
    done = existing_parity(path, NEW_COLUMN)
    print(f"{cond:9s} {len(g):3d} groups | prompt ~tokens p50 {lens[len(lens)//2]:,} "
          f"max {lens[-1]:,} | already done {len(done)}")

explicit  250 groups | prompt ~tokens p50 2,054 max 2,836 | already done 4
implicit  250 groups | prompt ~tokens p50 3,628 max 5,135 | already done 4


## 3. The prompt

Read one in full before spending GPU time — this text is the entire intervention.

In [14]:
g = groups["explicit"][0]
demo = build_user_prompt(g["Topic"], g["Source_Law"],
                         g["High Literacy"], g["Mid Literacy"], g["Low Literacy"])
print("SYSTEM:", SYSTEM_PROMPT)
print("\nUSER (instructions section only; the three responses are spliced in above it):\n")
print(demo[demo.index("TASK"):])

SYSTEM: You rewrite expert explanations for readers with less background knowledge. You never remove factual content to achieve simplicity. You achieve it by shortening sentences and choosing common words for everything except the technical terms themselves.

USER (instructions section only; the three responses are spliced in above it):

TASK
Write a new MID response and a new LOW response.

Take ALL factual content from the HIGH response. The existing MID and LOW responses show
you the target reading level only; they are incomplete and you must not use them as your
source of facts.

PRESERVE - every one of these that appears in the HIGH response must appear in both the new
MID and the new LOW response, with its exact value:
  - law, regulation, directive, treaty and standard names
  - article, section, clause, recital and paragraph numbers
  - deadlines, durations, time limits and notice periods
  - monetary amounts, penalties, fines and fee thresholds
  - percentages, quantities, lim

## 4. Load the model

In [15]:
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model_kwargs = dict(dtype=torch.bfloat16, device_map="auto")
if LOAD_IN_4BIT:
    from transformers import BitsAndBytesConfig
    model_kwargs.pop("dtype")
    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
    )

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **model_kwargs)
model.eval()

EOS_IDS = {tok.eos_token_id}
_cfg = getattr(model.generation_config, "eos_token_id", None)
EOS_IDS.update(_cfg if isinstance(_cfg, (list, tuple, set)) else [_cfg])
_im = tok.convert_tokens_to_ids("<|im_end|>")
if isinstance(_im, int) and _im >= 0:
    EOS_IDS.add(_im)
EOS_IDS.discard(None)

print(f"loaded | {sum(p.numel() for p in model.parameters())/1e9:.2f}B params")
if torch.cuda.is_available():
    print(f"vram: {torch.cuda.memory_allocated()/1024**3:.2f} GiB")

Loading weights: 100%|██████████| 398/398 [00:05<00:00, 73.72it/s] 
Some parameters are on the meta device because they were offloaded to the cpu.


loaded | 4.02B params
vram: 6.40 GiB


## 5. Generation and JSON handling

`parse_response` is deliberately tolerant — it recovers from code fences, surrounding prose, raw
newlines inside string values, unescaped inner quotes, trailing commas, and a fallback to `<MID>`/`<LOW>`
tags. It is unit-tested in `src/parity_rewrite.py` against all of those.

Each attempt is scored by entity-proxy coverage against the High response. An attempt below
`COVERAGE_MIN` is retried at a lower temperature, and the **best attempt is kept** across passes, so a
hard event yields a mediocre rewrite rather than nothing.

In [16]:
def log(msg=""):
    print(msg, flush=True)


def generate_batch(batch_groups, temperature, notes=None):
    # One fresh session per event group; batching is concurrency, not shared context.
    # `notes` carries per-group corrective feedback from the previous failed attempt.
    notes = notes or {}
    texts = []
    for g in batch_groups:
        user = build_user_prompt(g["Topic"], g["Source_Law"],
                                 g["High Literacy"], g["Mid Literacy"], g["Low Literacy"])
        user += notes.get(g["ID"], "")
        texts.append(tok.apply_chat_template(
            [{"role": "system", "content": SYSTEM_PROMPT},
             {"role": "user", "content": user}],
            tokenize=False, add_generation_prompt=True,
        ))
    enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
    with torch.inference_mode():
        out = model.generate(
            **enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=temperature > 0,
            temperature=temperature or None, top_p=GEN_TOP_P, top_k=GEN_TOP_K,
            pad_token_id=tok.pad_token_id,
        )
    completions = out[:, enc["input_ids"].shape[1]:]
    return tok.batch_decode(completions, skip_special_tokens=True)


def score(group, parsed):
    # Two gates now. Entity coverage protects substance; the tone gate protects register --
    # the first batch passed coverage while collapsing the Style axis from 0.08 to 0.86.
    if not parsed:
        return {"cov": -1.0, "covs": None, "bad_tone": ["MID", "LOW"], "accept": False,
                "tone": (0.0, 0.0)}
    cm = coverage(group["High Literacy"], parsed["mid"])
    cl = coverage(group["High Literacy"], parsed["low"])
    tm, tl = informality_score(parsed["mid"]), informality_score(parsed["low"])
    bad = []
    if not tone_ok(parsed["mid"], "Mid Literacy"):
        bad.append("MID")
    if not tone_ok(parsed["low"], "Low Literacy"):
        bad.append("LOW")
    cov = min(cm, cl)
    return {"cov": cov, "covs": (cm, cl), "bad_tone": bad, "tone": (tm, tl),
            "accept": cov >= COVERAGE_MIN and not bad}


def rank(s):
    # ordering key for "best attempt so far": accepted first, then coverage, then tone
    return (s["accept"], s["cov"], -len(s["bad_tone"]))

### Run loop

Resumable, checkpointed and OOM-tolerant, following the same pattern as notebook 00. An event is
"done" when both its Mid and Low rows carry a rewrite, so re-running the cell continues where it
stopped and regenerates nothing.

In [17]:
def run_condition(cond):
    path = FILES[cond]
    all_groups = groups[cond]

    done_ids = existing_parity(path, NEW_COLUMN)
    todo = [g for g in all_groups if g["ID"] not in done_ids]
    if SMOKE_TEST:
        todo = todo[:SMOKE_ROWS]
    if not todo:
        log(f"{cond}: nothing to do ({len(done_ids)} groups already rewritten)")
        return {}

    log("")
    log(f"=== {cond} | {len(todo)} groups | batch {BATCH_SIZE} | "
        f"coverage>={COVERAGE_MIN} | tone>={TONE_MIN} | max {MAX_PASSES} passes ===")

    best, results = {}, {}
    torch.manual_seed(SEED)
    t0 = time.time()

    pending = list(todo)
    notes = {}
    for pass_no in range(MAX_PASSES):
        if not pending:
            break
        temp = round(GEN_TEMPERATURE if pass_no == 0 else max(0.1, GEN_TEMPERATURE - 0.1 * pass_no), 2)
        log(f"\n-- pass {pass_no + 1}/{MAX_PASSES} | {len(pending)} groups | temperature {temp}")

        i, batch, n_since = 0, BATCH_SIZE, 0
        next_pending, next_notes = [], {}
        try:
            while i < len(pending):
                chunk = pending[i:i + batch]
                t_b = time.time()
                try:
                    raws = generate_batch(chunk, temp, notes)
                except torch.cuda.OutOfMemoryError:
                    torch.cuda.empty_cache(); gc.collect()
                    if batch == 1:
                        raise
                    batch = max(1, batch // 2)
                    log(f"  ! CUDA OOM -> batch={batch}, retrying")
                    continue

                dt = max(time.time() - t_b, 1e-9)
                for grp, raw in zip(chunk, raws):
                    parsed = parse_response(raw)
                    s = score(grp, parsed)
                    if grp["ID"] not in best or rank(s) > rank(best[grp["ID"]][0]):
                        best[grp["ID"]] = (s, parsed)

                    if s["accept"]:
                        results[grp["ID"]] = parsed
                    else:
                        next_pending.append(grp)
                        next_notes[grp["ID"]] = retry_note(s["cov"] >= COVERAGE_MIN, s["bad_tone"])

                    if LOG_EVERY_ROW:
                        if parsed is None:
                            log(f"  {grp['ID']:9s} PARSE FAILED  {raw.strip()[:60]!r}")
                        else:
                            cm, cl = s["covs"]
                            tm, tl = s["tone"]
                            flag = "ok  " if s["accept"] else ("cov " if s["cov"] < COVERAGE_MIN else "") + \
                                   ("tone:" + "/".join(s["bad_tone"]) if s["bad_tone"] else "")
                            log(f"  {grp['ID']:9s} {flag:12s} cov {cm:.2f}/{cl:.2f} tone {tm:4.1f}/{tl:4.1f} "
                                f"fk {fk_grade(parsed['mid']):4.1f}/{fk_grade(parsed['low']):4.1f} | "
                                f"{parsed['low'][:PREVIEW_CHARS]}")

                i += len(chunk); n_since += len(chunk)
                rate = i / max(time.time() - t0, 1e-9)
                log(f"  -- {len(chunk)} in {dt:5.1f}s | {i}/{len(pending)} this pass | "
                    f"eta {(len(pending)-i)/max(rate,1e-9)/60:5.1f}m")

                if n_since >= CHECKPOINT_EVERY:
                    # checkpoint the best attempt for every group seen so far, not just the
                    # accepted ones -- a crash mid-pass otherwise loses all pending work
                    snapshot = dict(results)
                    for gid, (s_b, p_b) in best.items():
                        if p_b:
                            snapshot.setdefault(gid, p_b)
                    write_parity_column(path, snapshot, NEW_COLUMN, SOURCE_COLUMN)
                    log(f"  .. checkpoint: {len(results)} accepted, {len(snapshot)} written")
                    n_since = 0
        except KeyboardInterrupt:
            snapshot = dict(results)
            for gid, (s_b, p_b) in best.items():
                if p_b:
                    snapshot.setdefault(gid, p_b)
            write_parity_column(path, snapshot, NEW_COLUMN, SOURCE_COLUMN)
            log(f"\ninterrupted -- {len(snapshot)} groups saved; re-run to resume")
            return snapshot

        pending, notes = next_pending, next_notes

    salvaged = 0
    for grp in pending:
        s_b, p_b = best.get(grp["ID"], (None, None))
        if p_b:
            results[grp["ID"]] = p_b
            salvaged += 1

    n = write_parity_column(path, results, NEW_COLUMN, SOURCE_COLUMN)
    log(f"\ndone: {len(results)} groups ({n} cells) -> {os.path.relpath(path, ROOT)} "
        f"in {(time.time()-t0)/60:.1f} min")
    log(f"      accepted on both gates: {len(results)-salvaged} | salvaged below a gate: {salvaged}")
    failed = [g['ID'] for g in pending if not best.get(g['ID'], (None, None))[1]]
    if failed:
        log(f"      {len(failed)} never parsed, left empty: {failed[:8]}")
    return results

### Reset before regenerating

The first batch cleared the coverage gate but collapsed the Style axis — Low-literacy
responses scored 0.86 on *formal and professional*, against 0.08 for the originals. This run
regenerates with the tone requirements and the tone gate in place.

The existing `parity_answer` is copied to `parity_answer_v1` first, so the v1/v2 comparison
stays available, and then cleared — otherwise the resume logic sees 250 finished groups and
does nothing.

In [ ]:
RESET = True          # set False when resuming an interrupted regeneration

if RESET:
    for cond in ("explicit",):        # implicit is untouched
        n = reset_column(FILES[cond], NEW_COLUMN, backup_to=BACKUP_COLUMN)
        print(f"{cond}: backed up {NEW_COLUMN} -> {BACKUP_COLUMN} and cleared ({n} rows)")
        print(f"  groups now pending: {len(groups[cond]) - len(existing_parity(FILES[cond], NEW_COLUMN))}")
else:
    for cond in ("explicit",):
        print(f"{cond}: {len(existing_parity(FILES[cond], NEW_COLUMN))}/{len(groups[cond])} already done")

## 6. Explicit condition

250 groups. Re-run to resume.

In [18]:
res_explicit = run_condition("explicit")


=== explicit | 246 groups | batch 2 | coverage>=0.85 | max 3 passes ===

-- pass 1/3 | 246 groups | temperature 0.3
  CON_05    low cov mid 1.00 low 0.67 | fk  8.0/ 4.1 |  109/  83w | Filing Deadline is 30 days.

From Section [X] of the Consumer Act 2009.

It appl
  CON_06    low cov mid 0.38 low 0.38 | fk  7.8/ 7.8 |  158/ 158w | Product Recall is under EU General Product Safety Directive 2001/95/EC.

Article
  -- 2 in  82.2s | 2/246 this pass | eta 167.2m
  CON_07    ok  cov mid 1.00 low 1.00 | fk  4.9/ 2.7 |  131/  92w | Weight Deception means giving less than said. 
Example: 1 kg rice given as 800 g
  CON_08    low cov mid 0.50 low 0.50 | fk  5.7/ 4.9 |  181/ 155w | False advertising means lying in ads.

The Consumer Act 2009 bans it in Banglade
  -- 2 in 101.4s | 4/246 this pass | eta 185.2m
  CON_09    ok  cov mid 1.00 low 1.00 | fk  8.7/ 7.8 |  170/ 163w | Subscription auto-renewal means a service keeps going without your action.

EU C
  CON_10    low cov mid 0.75 low 0.75 | fk

## 7. Implicit condition

250 groups. Longer prompts — drop `BATCH_SIZE` to 1 if this OOMs repeatedly.

In [20]:
res_implicit = run_condition("implicit")


=== implicit | 244 groups | batch 2 | coverage>=0.85 | max 3 passes ===

-- pass 1/3 | 244 groups | temperature 0.3

interrupted -- 0 groups saved; re-run to resume


## 8. Did it work?

The test is a **dissociation**, not an improvement on every axis:

* entity coverage should be flat and near 1.0 across Mid and Low
* FK grade should still descend High > Mid > Low
* if FK stopped descending, the preservation instruction beat the register instruction and you have
  produced three High-literacy responses — a worse model, which will still look like success on the
  content metrics

In [ ]:
import csv as _csv
_csv.field_size_limit(min(sys.maxsize, 2**31 - 1))

def verify(cond):
    path = FILES[cond]
    with open(path, newline="", encoding="utf-8") as fh:
        rows = list(_csv.DictReader(fh))
    cols = list(rows[0].keys())
    if NEW_COLUMN not in cols:
        print(f"=== {cond} ===\n  no {NEW_COLUMN} column yet -- run the generation cell first\n")
        return
    assert cols.index(NEW_COLUMN) == cols.index(SOURCE_COLUMN) + 1, "column misplaced"

    by_id = {}
    for r in rows:
        by_id.setdefault(r["ID"], {})[r["Literacy_Level"]] = r
    complete = [d for d in by_id.values()
                if all((d.get(l, {}).get(NEW_COLUMN) or "").strip() for l in LEVELS)]

    print(f"=== {cond} ===")
    print(f"  {NEW_COLUMN} sits directly after {SOURCE_COLUMN} | complete groups {len(complete)}/{len(by_id)}")
    if not complete:
        return

    print(f"  {'level':16s}{'words':>8s}{'FK':>7s}{'entity cov':>12s}")
    for lvl in LEVELS:
        w  = [len(d[lvl][NEW_COLUMN].split()) for d in complete]
        fk = [fk_grade(d[lvl][NEW_COLUMN]) for d in complete]
        cv = [coverage(d["High Literacy"][SOURCE_COLUMN], d[lvl][NEW_COLUMN]) for d in complete]
        print(f"  {lvl:16s}{sum(w)/len(w):8.1f}{sum(fk)/len(fk):7.1f}{sum(cv)/len(cv):12.2f}")

    fks = [sum(fk_grade(d[l][NEW_COLUMN]) for d in complete) / len(complete) for l in LEVELS]
    cvs = [sum(coverage(d["High Literacy"][SOURCE_COLUMN], d[l][NEW_COLUMN]) for d in complete) / len(complete)
           for l in LEVELS]
    base = [sum(coverage(d["High Literacy"][SOURCE_COLUMN], d[l][SOURCE_COLUMN]) for d in complete) / len(complete)
            for l in LEVELS]

    print(f"\n  FK descends High>Mid>Low : {'YES' if fks[0] > fks[1] > fks[2] else 'NO -- register adaptation lost'}")
    print(f"  entity coverage before   : {base[1]:.2f} (mid) {base[2]:.2f} (low)")
    print(f"  entity coverage after    : {cvs[1]:.2f} (mid) {cvs[2]:.2f} (low)")
    print(f"  DISSOCIATION ACHIEVED    : "
          f"{'YES' if fks[0] > fks[1] > fks[2] and cvs[2] > base[2] + 0.1 else 'not yet'}\n")

for c in FILES:
    verify(c)

=== explicit ===
  parity_answer sits directly after cleaned_answer | complete groups 4/250
  level              words     FK  entity cov
  High Literacy      574.2   15.6        1.00
  Mid Literacy       282.8   10.3        0.83
  Low Literacy       217.0    8.6        0.71

  FK descends High>Mid>Low : YES
  entity coverage before   : 0.33 (mid) 0.43 (low)
  entity coverage after    : 0.83 (mid) 0.71 (low)
  DISSOCIATION ACHIEVED    : YES

=== implicit ===
  parity_answer sits directly after cleaned_answer | complete groups 4/250
  level              words     FK  entity cov
  High Literacy      840.8   13.3        1.00
  Mid Literacy       422.0    9.0        0.66
  Low Literacy       348.5    6.6        0.66

  FK descends High>Mid>Low : YES
  entity coverage before   : 0.26 (mid) 0.25 (low)
  entity coverage after    : 0.66 (mid) 0.66 (low)
  DISSOCIATION ACHIEVED    : YES



## 9. Next

* Turn off `SMOKE_TEST`, read the four smoke outputs first, then run both conditions in full (500 calls).
* The entity check here is a free regex proxy. Before trusting these as training targets, re-run the
  paper's rubric extractor over `parity_answer` — that is the number the audit actually reports.
* Then point the fine-tuning at this column. Note that `make_sft_records` in `src/audit_utils.py`
  currently builds targets from the *historical* responses, which exhibit the stratification; training on
  those teaches the model to stratify harder. Switch the assistant turn to `parity_answer` for the
  improvement direction.